# Phase 6 evaluation: does bundle adjustment actually reduce trajectory error?

Every earlier Phase 6 test (`phase6_bundle_adjust`, `phase6_best_frame`) ran
against `Video1.avi`, which has no ground truth -- "the separated branch is
gone" was a visual read, not a number. UnityCam's synthetic test split has
real GT pose, so this runs `reconstruct_video()` twice on the same frames --
`bundle_adjust_enabled=False` then `True` -- and scores both against GT with
the exact same ATE/RPE metrics Phase 5 used (`src/eval/metrics.py`), via the
new `src/eval/run_ba_comparison.py::compare_bundle_adjustment()`.

Both conditions run the identical pipeline (DarkIR-lite -> Mini-3D-Recon ->
ICP chain [-> bundle adjustment]), so DarkIR-lite running on already-clean
synthetic frames doesn't bias the comparison -- it's an isolated A/B on the
bundle-adjustment flag alone, not a rerun of Phase 5's dark-robustness test.

Same Pascal/P100 torch-kernel fix as every other GPU notebook here.


## 0. Setup: clone repo + DarkIR upstream, reinstall Pascal-compatible torch, install deps

In [ ]:
REPO_URL = "https://github.com/ritiksharma3/endoslam.git"
DARKIR_URL = "https://github.com/cidautai/DarkIR.git"

!git clone $REPO_URL repo
!git clone $DARKIR_URL repo/DarkIR_upstream
%cd repo

!pip install -q torch==2.5.1 torchvision==0.20.1 --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q -r environment/requirements.txt


## 1. GPU check + resolve dataset root and both checkpoints

In [ ]:
import os
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

gpu_actually_usable = False
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    try:
        torch.zeros(1, device="cuda") + torch.zeros(1, device="cuda")
        gpu_actually_usable = True
        print("GPU FIX CONFIRMED: real CUDA op succeeded")
    except RuntimeError as e:
        print(f"GPU still not usable after torch reinstall: {e}")


def find_endoslam_root(base="/kaggle/input", max_depth=4):
    for root, dirs, _files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        if os.path.basename(root).lower() == "endoslam":
            return root
    return None


def find_phase_checkpoint(name_fragment: str, base="/kaggle/input"):
    candidates = [os.path.join(r, f) for r, _, fs in os.walk(base) for f in fs
                  if f.startswith("epoch_") and f.endswith(".pt") and name_fragment in r.lower()]
    if not candidates:
        return None
    return max(candidates, key=lambda p: int(os.path.basename(p).split("_")[1].split(".")[0]))


DATA_ROOT = find_endoslam_root()
assert DATA_ROOT, "could not find an endoslam dir under /kaggle/input"
print("DATA_ROOT:", DATA_ROOT)

DARKIR_CHECKPOINT_PATH = find_phase_checkpoint("darkir")
assert DARKIR_CHECKPOINT_PATH, "could not find a Phase 2 DarkIR-lite checkpoint under /kaggle/input"
print("DARKIR_CHECKPOINT_PATH:", DARKIR_CHECKPOINT_PATH)

MINI_RECON_CHECKPOINT_PATH = find_phase_checkpoint("mini3drecon")
assert MINI_RECON_CHECKPOINT_PATH, "could not find a Phase 3 Mini-3D-Recon checkpoint under /kaggle/input"
print("MINI_RECON_CHECKPOINT_PATH:", MINI_RECON_CHECKPOINT_PATH)


## 2. Load config, test-split dataset, and both trained models

In [ ]:
import yaml

from src.common.device import select_device
from src.data.endoslam_dataset import EndoSLAMStomachDataset
from src.darkir_lite.model import build_darkir_lite
from src.reconstruction.model import MiniReconModel

with open("configs/config.yaml") as f:
    config = yaml.safe_load(f)
config["data"]["root"] = DATA_ROOT

device = select_device()
print("device:", device)

# test split -- held out from both Phase 2 and Phase 3 training, same split
# Phase 5's evaluation used.
dataset = EndoSLAMStomachDataset(config, split="test", cameras=["UnityCam"],
                                  context_window=config["reconstruction"]["context_window"])
print(f"UnityCam test windows: {len(dataset)}")

darkir_model = build_darkir_lite(pretrained=False).to(device)
darkir_checkpoint = torch.load(DARKIR_CHECKPOINT_PATH, map_location=device, weights_only=False)
darkir_model.load_state_dict(darkir_checkpoint["model_state_dict"])
darkir_model.eval()
print(f"loaded DarkIR-lite: epoch={darkir_checkpoint['epoch']}, val_psnr={darkir_checkpoint.get('val_psnr')}")

mini_recon_model = MiniReconModel(
    pretrained=False, depth_head_channels=config["reconstruction"]["depth_head_channels"]
).to(device)
mini_recon_checkpoint = torch.load(MINI_RECON_CHECKPOINT_PATH, map_location=device, weights_only=False)
mini_recon_model.load_state_dict(mini_recon_checkpoint["model_state_dict"])
mini_recon_model.eval()
print(f"loaded Mini-3D-Recon: epoch={mini_recon_checkpoint['epoch']}, "
      f"val_depth_absrel={mini_recon_checkpoint.get('val_depth_absrel')}")


## 3. Run both conditions (bundle adjustment off vs on)

`MAX_FRAMES = 200` bounds runtime -- `reconstruct_video()` runs its full
ICP-chain(-plus-BA) pass twice here, and the test split's exact size wasn't
known in advance.

In [ ]:
from src.eval.run_ba_comparison import compare_bundle_adjustment

MAX_FRAMES = 200

results = compare_bundle_adjustment(
    dataset, darkir_model, mini_recon_model, config, device, max_frames=MAX_FRAMES,
)

for condition, r in results.items():
    print(f"
--- {condition} ---")
    print(r)


## 4. Save metrics JSON

In [ ]:
import json

os.makedirs("/kaggle/working/output", exist_ok=True)
METRICS_PATH = "/kaggle/working/output/phase6_ba_metrics.json"

with open(METRICS_PATH, "w") as f:
    json.dump(results, f, indent=2)
print("saved:", METRICS_PATH)
print(json.dumps(results, indent=2))


## Done

Download `output/phase6_ba_metrics.json`. Compare `ATE` / `RPE_trans_rmse` /
`RPE_rot_rmse_deg` between `icp_only` and `bundle_adjusted` -- lower is
better for all three. This is the first Phase 6 result with actual ground
truth behind it, not a visual read.